# HW1: Building a Gymnasium Environment for Underwater Caldera Exploration



Our setting is an underwater caldera exploration mission, in which an autonomous vehicle navigates through a volcanic basin and collects scientific measurements from different locations.

The vehicle operates under realistic physical and operational constraints. It has limited fuel, which restricts how far it can travel. In addition, when the vehicle moves on the ocean surface, its motion may be influenced by water currents, introducing uncertainty into its movement.

In this assignment, we will create a simulated environment by specifying:

the state of the system,
the actions available to the vehicle,
the dynamics governing movement and uncertainty,
and the reward function that encodes the mission setting.

This simulation provides a structured representation of the environment, enabling systematic interaction and analysis of the system under different conditions. It will allow us to later study and design algorithms for planning and decision-making in challenging exploration settings.

<img src="../images/Kolumbo-submarine-volcano-and-caldera-Alexandri-et-al.jpg" alt="Kolumbo submarine caldera" width="500" />


When considering autonomous agents, we need to consider the environment with which they interact: Agents choose actions to perform and recieve from the environment observations and rewards. In this assignment, we focus on the **environment**, and on modeling the exploration task. In later assignments, we will switch focus to the agent.

<img src="../images/agent-env.png" alt="agent-env" width="500" />



In this assignment, we are building a custom [Gymnasium](https://gymnasium.farama.org/index.html) environment for an underwater caldera exploration mission. We are using Gymnasium here because it gives our underwater exploration problem a clean, reusable RL interface that will work well for simulation, debugging, training, and evaluation.

That means we need to define:
- what the agent observes about the mission state
- what actions it can take while exploring
- what reward it receives for performing actions
- when an episode should end



In [ ]:
%load_ext autoreload
%autoreload 2

import unittest

import gymnasium as gym
from gymnasium import spaces
from IPython.display import Markdown
import matplotlib.pyplot as plt
import numpy as np

import answers
from caldera_env import *
from to_implement import *

## Mission story


The robotic agent in our setting operates on the water surface. The environment is therefore modeled as a 2D grid laid over an underwater caldera.

At each time step, the agent either moves at any of the perpendicular direction or takes a sample at its current location.

Depending on the task, some locations may be more scientifically valuable than others. For example, the agent may seek to identify the point of maximal depth.

Importantly, different objectives can be naturally expressed through different formulations of the reward function, allowing the same environment to support a variety of scientific goals. This is known as **reward shaping**, the practice of designing or augmenting the reward signal to guide the agent toward desired behaviors while preserving the underlying task objectives.

### Modeling the Exploration Task


We design the exploration setting in terms of two separate layers: one for representing the environment, and the second for representing the exploration process.


For the first layer, we define a continuous field that represents the map environment and its underlying phenomenon as a function
$
f : \mathbb{R}^2 \rightarrow \mathbb{R}.
$
As is common in spatial modeling, we use a **mixture of Gaussians** to capture structured variability across space:

$$
f(x,y) = \sum_{i=1}^{K} w_i \exp\left(
-\left[
\frac{(x - \mu_{x,i})^2}{2\sigma_{x,i}^2}
+
\frac{(y - \mu_{y,i})^2}{2\sigma_{y,i}^2}
\right]
\right)
$$


A mixture of Gaussians provides a smooth, interpretable basis for modeling heterogeneous spatial structure in the caldera and can represent quantities such as depth or chemical concentration. In our example, we use it to represent localized pits that arise due to non-uniform collapse of the caldera floor, hydrothermal activity, and secondary vent formation. Importantly, it is defined independently of any particular grid or discretization. It is within this abstraction that we represent the agent's movements and observations.

Below is a visualization of an example environment with three Gaussian-induced pits, with the agent marked by a black dot. 


<img src="../images/caldera-map.png" alt="agent-env" width="500" />


The second layer is an abstraction introduced to support the exploration task. In this discretization layer, the grid is a sampling scheme over a continuous latent field, and samples are collected at discrete locations. This means that each cell $(𝑥,𝑦)$ corresponds to a point at which the field can be evaluated (in our implementation, the bottom-left corner of the cell). This discrete resolution over the continuous space corresponds to real-world sensing limitations, where measurements can only be obtained at a finite set of locations.

The grid can have different granularities.

<img src="../images/caldera-map-grid.png" alt="agent-env" width="500" />

💡
**This separation decouples environment semantics (what the world represents) from numerical resolution (how finely it is sampled). As a result, different grid resolutions can represent the same underlying environment, with only the sampling resolution changing.** 


In this assignment, we consider three settings of increasing complexity:

*1*. a **deterministic environment**, with deterministic effects of movement and sampling actions

*2*. a **stochastic environment**, where actions may fail (e.g., due to the effect of ocean currents)

*3*. a **partially observable environment**, where the agent has a limited ability to perceive other vehicles in the environment but must avoid collisions

Remember that across all these settings, the agent may be assigned different missions, captured through different reward functions. This allows us to study how task definition influences agent behavior.


## General Guidelines

While we provide the complete framework for your implementation, the only files you will submit are:

- to_implement.py - where you implement the coding assignment
- answers.py - where you answer the theoretical questions


⚠️ Warning: These will be the only files that will be examined - so any change you make to other files will not be considered.  

## Instructions 

###  The caldera environment


In caldera_env.py you can find a basic implementation of the environment, called **BaseCalderaEnv**.

The map of the environment is defined by the dimensions X,Y (dim_x, dim_y). The geological phenomena are represented as pits, and are given as input by the pit_params, representing the parameters of the pit distributions, and pit_weights, representing how deep each pit is. The distribution of each pit is defined in **bivariate_normal(params: BivariateNormalStruct)**, and the **_caldera_sim_function** generates the entire map.

 
The agent is a vehicle within the environment. For simplicity, the agent is represented as a point, defined by its x,y dimension on the map. The agent's initial position is given as input, as well as its movement size (in the map dimensions). The agent has an energy level, which is consumed by actions according to the specified per-action type values.  

The sampling resolution is defined by sampling_res, which in effect defines a sampling grid over the map.

In addition to the agent, there may be other vehicles in the environment, which may be defined at initialization or added/removed later on. If the agent tries to move into an occupied cell or beyond the map boundaries, it may either cause the episode to end (if end_episode_on_collision is True), or (if end_episode_on_collision is False) leave the agent in place.

        


An example of an environment with other vehicles, can be found below. 

<img src="../images/caldera-map-grid-with-obstacles.png" alt="agent-env" width="500" />

**Actions** 

The agent moves at each time step, choosing between moving at any of the perpendicular direction or taking a sample at its current location.

Our action space is discrete and is therefore encoded as `spaces.Discrete(5)`.


In [ ]:
action_space = spaces.Discrete(5)

**Observation space**


A valuable observation should contain the information the agent needs to make a decision. 
In gymnasium, the observation space is defined using spaces, as in the example below. 

In [ ]:
example_observation_space = spaces.Dict(
    {
        "position": spaces.Box(low=0, high=4, shape=(2,), dtype=int),
        "resource": spaces.Discrete(5),
    }
)

In our setting, the observation can include the agent’s position, its remaining energy, whether the current cell has already been sampled, and the value collected at the current position.

The environment itself may maintain additional state information, such as a map of visited cells and their collected values, as well as the locations of other vehicles. This information is not necessarily directly available to the agent; rather, it is the responsibility of the environment designer to decide which components of the state are revealed through the observation.

Under partial observability, the observation must also include information about obstacles, restricted to what the agent can perceive within its field of view.

You will implement several observation functions corresponding to different observability assumptions.

**Reward**



Reward is where the mission objective becomes explicit.

A simple reward function could be:

- small penalty for movement, to represent energy cost
- larger positive reward for sampling a valuable unsampled cell
- small penalty for sampling a location twice


The reward sets what the agent learns. For example, we may want it to encourage the agent to learn that the best policy is not just to move, but to move toward useful places and sample selectively.

You will implement a reward function corresponding to a specified objective.

## Task 1: Creating the basic environment

In BaseCalderaEnv we have implemented a lot of the functionality. However, several methods are missing, and will throw an exception if they are used. 

As a first step, you will complete the implementation of the basic environment, named **CalderaEnv** in which the actions are deterministic and the environment is fully observable.

You will implement four functions:

    def _get_observation_space
    def _get_observation
    def _get_sample
    def _perform_move

and two reward functions

        def reward_function_explore


        def reward_function_gap_to_max



**_get_observation** and **_get_observation_space**

An observation needs to include:

* "position": the current position of the agent  (spaces.Box)            
* "energy": the current energy level (spaces.Discrete)
* "sampled_before": flag indicating if the cell has been sampled before (spaces.Discrete(2))
* "value": the value at the current cell (spaces.Box).

If the current position has been sampled, value is the value of the current cell. Otherwise, value is the default value (DEFAULT_VALUE).

_get_observation_space needs to be adapted to the specified structure. 


**_get_sample**

Get the value of the current cell and whether it has been sampled before.
Update the value in *self.sampled_cells[sampling_grid_cell]* and update the max and min values observed so far. 


**_perform_move**

 Perform the movement action by generating the path to the proposed destination, according to movement_size.
 Check for collisions along the path, and stop when and if a collistion occurs. 


 Return  **position, collision_occurred**, the final position after attempting the move (and progressing until a collision occurs, if there is one), and the flag indicating if a collision occurred.


        
**reward_function_explore** and **reward_function_gap_to_max**

Implement the reward functions that will support the exploration task.

**reward_function_explore** : provides a positive reward for sampling new cells and a negative reward for sampling previously sampled cells or taking movement actions.

**reward_function_gap_to_max**: provides a positive reward based on the value of the current cell relative to the maximum value observed so far. As in reward_function_explore, a negative reward is returned for sampling previously sampled cells or taking movement actions.







⚠️ Note: There are multiple valid ways to implement the reward functions. At this stage, your implementation will be evaluated based on compliance with the specified requirements, not on the overall performance achieved.

### Testing your implementation


#### 1) Core checks first (fast + deterministic)
Before plotting anything, we run concise unit-style checks to validate the required behavior:
- observation fields and default value handling,
- sampling logic (`sampled_before`, sampled value caching, min/max updates),
- collision-aware movement, and
- both reward functions.


In [ ]:
test = unittest.TestCase()
np.random.seed(7)

# --- Functional checks ---
env = CalderaEnv(
    dim_x=20,
    dim_y=20,
    sampling_res=5,
    initial_position=(5, 5),
    movement_size=3,
    initial_energy=20,
    other_vehicles=[((8, 5), 0)],  # point obstacle directly to the east
    reward_function=reward_function_explore,
)

obs, _ = env.reset(seed=0)
test.assertSetEqual(set(obs.keys()), {"position", "energy", "sampled_before", "value"})
test.assertEqual(obs["sampled_before"], 0)
test.assertEqual(obs["value"], DEFAULT_VALUE)
test.assertTrue(env.observation_space.contains(obs))

obs_1, reward_1, *_ = env.step(SAMPLE)
energy_after_first_sample = obs_1["energy"]
test.assertEqual(obs_1["sampled_before"], 0)
test.assertAlmostEqual(reward_1, 1.0)
test.assertEqual(len(env.sampled_cells), 1)
test.assertAlmostEqual(env.get_max_value_observed(), obs_1["value"])
test.assertAlmostEqual(env.get_min_value_observed(), obs_1["value"])

obs_2, reward_2, *_ = env.step(SAMPLE)
test.assertEqual(obs_2["sampled_before"], 1)
test.assertAlmostEqual(reward_2, -0.5)
test.assertEqual(obs_2["energy"], energy_after_first_sample)

obs_3, reward_3, terminated_3, truncated_3, _ = env.step(MOVE_EAST)
test.assertEqual(tuple(obs_3["position"]), (7, 5))  # stopped before obstacle at (8, 5)
test.assertAlmostEqual(reward_3, -0.1)
test.assertFalse(terminated_3 or truncated_3)

# Gap-to-max reward sanity check
env_gap = CalderaEnv(
    dim_x=20,
    dim_y=20,
    sampling_res=5,
    initial_position=(0, 0),
    movement_size=5,
    initial_energy=20,
    reward_function=reward_function_gap_to_max,
)

env_gap.reset(seed=1)
obs_g1, reward_g1, *_ = env_gap.step(SAMPLE)
test.assertAlmostEqual(reward_g1, 0.0)  # first sample is the running max
env_gap.step(MOVE_NORTH)
obs_g2, reward_g2, *_ = env_gap.step(SAMPLE)
test.assertEqual(obs_g2["sampled_before"], 0)
test.assertLessEqual(reward_g2, 0.0)

print("✅ Task 1 core checks passed (API, sampling, movement, rewards).")


#### 2) Engaging visual check
Now that the core behavior is verified, we run a short trajectory and visualize:
- the agent path on the caldera map, and
- the reward timeline over the rollout.


In [ ]:
demo_env = CalderaEnv(
    dim_x=40,
    dim_y=30,
    sampling_res=5,
    initial_position=(0, 0),
    movement_size=5,
    initial_energy=50,
    other_vehicles=[((15, 10), 5)],
    reward_function=reward_function_explore,
)

demo_env.reset(seed=2)
demo_actions = [SAMPLE, MOVE_EAST, SAMPLE, MOVE_EAST, MOVE_NORTH, SAMPLE, MOVE_NORTH, MOVE_EAST]
reward_trace = []

for action in demo_actions:
    _, reward, *_ = demo_env.step(action)
    reward_trace.append(reward)

fig, _ = demo_env.visualize(show_grid_lines=True, show_agent_path=True)
plt.title("Task 1 • Agent trajectory + sampled map")
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(range(1, len(reward_trace) + 1), reward_trace, marker="o", linewidth=2)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.title("Task 1 • Reward timeline")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.grid(alpha=0.3)
plt.show()

print("🎉 Task 1 visual test complete.")


### ❓ Question 1 ❓

Define a formal decision-making model that captures the environment you created in this task. Use mathematical notation. Write your answer in `answers.py:q1`.

In [ ]:
Markdown(answers.q1)

## Task 2: Creating the stochastic environment

As a next phase, we implement a stochastic environment by implementing




        def stochastic_effet_wrong_turn(env, action, success_probability=0.8)


 According to the specified success probability, the agent executes a "wrong turn" instead of the intended action,corresponding right-turn action (as specified by WRONG_TURN_ACTION)
 For example, an intended move north may result in moving east instead.

### Testing your implementation


#### 1) Core checks first (fast + statistical sanity)
We first validate transition correctness before plotting:
- deterministic edge behavior for success probability 1.0 and 0.0,
- unchanged `SAMPLE` action,
- empirical success-rate sanity around the expected 0.8.


In [ ]:
test = unittest.TestCase()
np.random.seed(42)

env = SCalderaEnv(
    dim_x=40,
    dim_y=40,
    sampling_res=5,
    initial_position=(20, 20),
    initial_energy=200,
    movement_size=1,
    stochastic_effet_function=stochastic_effet_wrong_turn,
)

# --- Deterministic edge cases ---
for action in (MOVE_NORTH, MOVE_EAST, MOVE_SOUTH, MOVE_WEST, SAMPLE):
    test.assertEqual(stochastic_effet_wrong_turn(env, action, success_probability=1.0), action)

for action in (MOVE_NORTH, MOVE_EAST, MOVE_SOUTH, MOVE_WEST):
    test.assertEqual(
        stochastic_effet_wrong_turn(env, action, success_probability=0.0),
        env.WRONG_TURN_ACTION[action],
    )

test.assertEqual(stochastic_effet_wrong_turn(env, SAMPLE, success_probability=0.0), SAMPLE)

# --- Empirical distribution check ---
intended_action = MOVE_NORTH
success_probability = 0.8
num_trials = 4000

effective_actions = np.array(
    [
        stochastic_effet_wrong_turn(env, intended_action, success_probability=success_probability)
        for _ in range(num_trials)
    ]
)
success_rate = float(np.mean(effective_actions == intended_action))

test.assertGreater(success_rate, 0.76)
test.assertLess(success_rate, 0.84)

print(f"✅ Task 2 core checks passed. Empirical success rate: {success_rate:.3f}")


#### 2) Engaging visual check
Once the stochastic effect is validated, we visualize it in two complementary ways:
- a bar chart of intended-vs-wrong-turn outcomes,
- a rollout path showing how stochasticity bends the trajectory.


In [ ]:
np.random.seed(42)
env = SCalderaEnv(
    dim_x=40,
    dim_y=40,
    sampling_res=5,
    initial_position=(20, 20),
    initial_energy=200,
    movement_size=1,
    stochastic_effet_function=stochastic_effet_wrong_turn,
)

intended_action = MOVE_NORTH
effective_actions = np.array(
    [stochastic_effet_wrong_turn(env, intended_action, success_probability=0.8) for _ in range(4000)]
)

labels = [intended_action, env.WRONG_TURN_ACTION[intended_action]]
counts = [
    int(np.sum(effective_actions == intended_action)),
    int(np.sum(effective_actions == env.WRONG_TURN_ACTION[intended_action])),
]

plt.figure(figsize=(7, 4))
bars = plt.bar(labels, counts, color=["#2ca02c", "#ff7f0e"])
plt.title("Task 2 • Intended vs wrong-turn outcomes")
plt.ylabel("Count")
plt.grid(axis="y", alpha=0.3)
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        int(bar.get_height()),
        ha="center",
    )
plt.show()

env.reset(seed=0)
rollout_actions = [MOVE_NORTH] * 20 + [MOVE_EAST] * 20
for action in rollout_actions:
    env.step(action)

fig, _ = env.visualize(show_grid_lines=True, show_agent_path=True)
plt.title("Task 2 • Example stochastic trajectory")
plt.show()

print("🎉 Task 2 visual test complete.")


### ❓ Question 2 ❓

Define a formal decision-making model that captures the environment you created in this task. Use mathematical notation. Write your answer in `answers.py:q2`.

In [ ]:
Markdown(answers.q2)

## Task 3: Creating the partially observable environment





Finally, we implement the partially observable version of the environment, POCalderaEnv

In a partially observable environment, the agent does not have access to invariant information about the environment (e.g., the full map, the location of obstacles, etc.). Therefore, we raise an error if there is an attempt to access invariant information in this environment via **get_invariant_information**. 


⚠️ Do not change this function! This will be validated.
   




In addition, you need to implement the three following methods

    def _get_surrounding_obstacles
    def _get_observation_space
    def _get_observation


**_get_surrounding_obstacles**

This is a key function for modeling partial observability.

For each of the 8 cardinal and intercardinal directions, the function checks whether there is an obstacle within the agent’s observability range. It should return an array of 8 booleans, where each entry corresponds to a direction and indicates whether an obstacle is detected (True if an obstacle is present, False otherwise).

Note that observability is determined by the observability distance, measured in map units. For example, if the agent is located at position [5,5] and the first occupied cell in a given direction is at [10,10], then the agent will detect this obstacle only if its observability distance is at least 5.

**_get_observation** and **_get_observation_space**

Add to the observation the information about the surrounding obstacles, i.e., by adding **observation_space.spaces["surrounding_obstacles"] = spaces.MultiBinary(8)**

Adjust the two functions accordingly.     

### Testing your implementation


#### 1) Core checks first (partial-observability guarantees)
We start by validating strict PO behavior:
- `surrounding_obstacles` appears in both observation and observation space,
- obstacle detection depends on `observability_distance`,
- `get_invariant_information()` is blocked,
- post-step obstacle observations stay consistent.


In [ ]:
test = unittest.TestCase()
center = (5, 5)

# Place one point obstacle in each direction at distance 2 from the agent.
directional_obstacles = [
    ((center[0] + 2 * dx, center[1] + 2 * dy), 0)
    for dx, dy in DIRECTION_STEPS.values()
]

# With distance 1, nothing should be detected (all obstacles are at distance 2).
env_near = POCalderaEnv(
    dim_x=10,
    dim_y=10,
    sampling_res=1,
    initial_position=center,
    observability_distance=1,
    other_vehicles=directional_obstacles,
)
obs_near, _ = env_near.reset(seed=0)
test.assertIn("surrounding_obstacles", obs_near)
test.assertEqual(obs_near["surrounding_obstacles"].shape, (8,))
test.assertTrue(np.all(obs_near["surrounding_obstacles"] == 0))

# With distance 2, all 8 directions should detect an obstacle.
env = POCalderaEnv(
    dim_x=10,
    dim_y=10,
    sampling_res=1,
    initial_position=center,
    observability_distance=2,
    other_vehicles=directional_obstacles,
)
obs, _ = env.reset(seed=0)
test.assertTrue(np.all(obs["surrounding_obstacles"] == 1))
test.assertIn("surrounding_obstacles", env.observation_space.spaces)

with test.assertRaises(AttributeError):
    env.get_invariant_information()

obs_after_move, *_ = env.step(MOVE_EAST)
np.testing.assert_array_equal(
    obs_after_move["surrounding_obstacles"],
    env._get_surrounding_obstacles(tuple(obs_after_move["position"])),
)

print("✅ Task 3 core checks passed (PO constraints + obstacle sensing).")


#### 2) Engaging visual check (minimal radar)
A compact visualization flow:
- Use regular `env.visualize(...)` to show the **full map** (near + far obstacles).
- Build a radar directly from `obs["surrounding_obstacles"]` (no extra scanning logic).
- The setup includes far obstacles that stay undetected, and one wide nearby obstacle detected in both **NORTH** and **NORTHEAST**.


In [ ]:
center = (10, 10)
observability_distance = 4

# Nearby obstacles (detectable).
detectable_obstacles = [
    ((10, 12), 3),  # intentionally large enough to be detected in NORTH and NORTHEAST
    ((13, 9), 2),   # EAST
    ((7, 7), 2),    # SOUTHWEST
]

# Far obstacles (present on map, outside sensing range).
far_obstacles = [
    ((10, 19), 2),
    ((19, 19), 2),
    ((19, 10), 2),
    ((19, 1), 2),
    ((10, 1), 2),
    ((1, 1), 2),
    ((1, 10), 2),
    ((1, 19), 2),
]

all_obstacles = detectable_obstacles + far_obstacles

env = POCalderaEnv(
    dim_x=22,
    dim_y=22,
    sampling_res=1,
    initial_position=center,
    observability_distance=observability_distance,
    other_vehicles=all_obstacles,
)
obs, _ = env.reset(seed=0)

# 1) Full map view (regular environment visualization).
fig, _ = env.visualize(show_grid_lines=True, show_agent_path=False)
plt.title("Task 3 • Full PO map (near + far obstacles)")
plt.show()

# 2) Minimal radar view built directly from observation.
direction_names = list(DIRECTION_STEPS.keys())
detections = obs["surrounding_obstacles"].astype(int)
angles = np.linspace(0, 2 * np.pi, len(direction_names), endpoint=False)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, polar=True)
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.set_xticks(angles)
ax.set_xticklabels(direction_names)
ax.set_ylim(0, 1.2)
ax.set_yticks([0, 1])
ax.set_yticklabels(["", ""])
ax.grid(alpha=0.3)

for angle, detected in zip(angles, detections):
    ax.plot(
        [angle, angle],
        [0, 1],
        color="#d62728" if detected else "#1f77b4",
        alpha=0.55 if detected else 0.15,
        linewidth=2,
    )

mask = detections.astype(bool)
ax.scatter(
    angles[mask],
    np.ones(np.sum(mask)),
    s=150,
    c="#d62728",
    edgecolors="white",
    linewidths=1.0,
    zorder=5,
)

ax.set_title("Task 3 • Radar from obs['surrounding_obstacles']")
plt.show()

# Small sanity checks based on observation only.
index_by_direction = {name: i for i, name in enumerate(direction_names)}
assert detections[index_by_direction["NORTH"]] == 1
assert detections[index_by_direction["NORTHEAST"]] == 1
assert int(np.sum(detections)) < len(detections)  # some obstacles are too far to be sensed

print(f"Detected directions from obs: {[name for name, v in zip(direction_names, detections) if v]}")
print(f"Detected {int(np.sum(detections))}/8 directions with {len(all_obstacles)} total obstacles on the map.")
print("🎉 Task 3 radar visual test complete.")


### ❓ Question 3 ❓

Define a formal decision-making model that captures the environment you created in this task. Use mathematical notation. Write your answer in `answers.py:q3`.

In [ ]:
Markdown(answers.q1)

## Task 4: Dry Questions

### Question 4: Caldera Markov Chain Exercise

Suppose a surface support vessel prepares **3 calibrated sampling cartridges** for the vehicle before a caldera survey. Each cartridge allows exactly **one successful `SAMPLE` action** at a scientifically valuable location.

At the end of each survey episode, the mission manager checks how many prepared cartridges remain and follows this policy for the next episode:

- If **all** prepared cartridges were used during the episode, then the vessel prepares **3 new cartridges**.
- If **even 1 cartridge remains unused**, then **no new cartridges are prepared**.

Therefore, at the **beginning of any episode**, the mission starts with either **1, 2, or 3** cartridges available.

Historical mission logs suggest that the vehicle needs about **1 scientifically useful sample per episode on average**, but the true sample demand is random. Depending on currents, the realized trajectory, and which high-value cells are encountered, a mission episode may require **0, 1, 2, 3, or more** successful `SAMPLE` actions.

Assume the episode-level demands $D_1,D_2,\dots$ are **independent and identically distributed**, with each $D_n$ **Poisson** with mean 1:

$$
P(D_n=k)=\frac{e^{-1}}{k!}, \qquad k=0,1,2,\dots
$$

Use the following numerical approximations:

- $P(D_n=0)=0.368$
- $P(D_n=1)=0.368$
- $P(D_n=2)=0.184$
- $P(D_n=3)=0.061$
- $P(D_n\ge 4)=0.019$

Let $X_n$ be the number of available sampling cartridges at the **beginning** of episode $n$.

For the state update, if the realized demand in episode $n$ satisfies $D_n \ge X_n$, then all available cartridges are used and the next episode begins with $X_{n+1}=3$. If $D_n < X_n$, then $X_{n+1}=X_n-D_n$. In Question 4(d), “demand exceeds the available number of cartridges” means the strict event $D_n > X_n$.


### ❓ Question 4(a) ❓

Explain why $\{X_n\}$ can be modeled as a **Markov chain** with state space $\{1,2,3\}$.

Write your answer in `answers.py:q4a`.


In [ ]:
Markdown(answers.q4a)

### ❓ Question 4(b) ❓

Determine the **transition matrix** for this Markov chain.

Write your answer in `answers.py:q4b`.


In [ ]:
Markdown(answers.q4b)

### ❓ Question 4(c) ❓

Find the **stationary distribution** of the chain.

Write your answer in `answers.py:q4c`.


In [ ]:
Markdown(answers.q4c)

### ❓ Question 4(d) ❓

Using the stationary distribution, compute the long-run probability that the **episode's sample demand exceeds the available number of cartridges**.

That is, compute the probability of the strict event $D_n > X_n$.

Write your answer in `answers.py:q4d`.


In [ ]:
Markdown(answers.q4d)

### ❓ Question 4(e) ❓

Interpret the result in the context of HW1: how often is the agent unable to complete all desired `SAMPLE` actions, and does this policy likely cause lost scientific value?

Write your answer in `answers.py:q4e`.


In [ ]:
Markdown(answers.q4e)

### Question 5: Dynamic Bayesian Networks (DBNs)

We continue with the same setup as in **Question 4**. To turn this into a **dynamic Bayesian network (DBN)**, introduce an observation variable
$$
O_n \in \{0,1\},
$$
where
- $O_n=1$ means that the mission log reports that **episode $n$ had insufficient cartridges**, i.e. the sample demand exceeded supply,
- $O_n=0$ means that the available cartridges were sufficient.

Assume the observation is perfectly accurate, so that
$$
O_n = 1 \iff D_n > X_n.
$$

Note the distinction between the transition rule and the observation rule:
- the next state resets to $3$ when **all available cartridges are used**, i.e. when $D_n \ge X_n$;
- the observation reports **insufficient cartridges** only when demand strictly exceeds supply, i.e. when $D_n > X_n$.

You do **not** need to draw a graph. Write all answers in MathJax/text using parent sets, factorizations, and conditional probability expressions.


### ❓ Question 5(a) ❓

Define in words the meaning of each of the following random variables:
$$
X_n, \qquad D_n, \qquad O_n, \qquad X_{n+1}.
$$
Explain which of these variables represent:
- the system state,
- a stochastic exogenous input,
- an observation.

Write your answer in `answers.py:q5a`.


In [ ]:
Markdown(answers.q5a)

### ❓ Question 5(b) ❓

Without drawing a graph, write the parent set of each variable in one 2-time-slice DBN.

That is, specify:
$$
\mathrm{Pa}(D_n), \qquad \mathrm{Pa}(O_n), \qquad \mathrm{Pa}(X_{n+1}).
$$

Write your answer in `answers.py:q5b`.


In [ ]:
Markdown(answers.q5b)

### ❓ Question 5(c) ❓

Using your parent sets, write the factorization of the one-step joint distribution
$$
P(X_n, D_n, O_n, X_{n+1}).
$$

Then write the factorization of the full joint distribution over $N$ episodes:
$$
P(X_1, D_1, O_1, X_2, D_2, O_2, \dots, X_N, D_N, O_N, X_{N+1}).
$$

Write your answer in `answers.py:q5c`.


In [ ]:
Markdown(answers.q5c)

### ❓ Question 5(d) ❓

Write $P(X_{n+1} \mid X_n, D_n)$ using the replenishment policy.

Your answer may be given as a deterministic rule, for example by specifying the value of $X_{n+1}$ in terms of $X_n$ and $D_n$.

Write your answer in `answers.py:q5d`.


In [ ]:
Markdown(answers.q5d)

### ❓ Question 5(e) ❓

Write $P(O_n \mid X_n, D_n)$ explicitly.

Because the observation is perfect, your answer should assign probability $1$ to the correct value of $O_n$ and probability $0$ otherwise.

Write your answer in `answers.py:q5e`.


In [ ]:
Markdown(answers.q5e)

### ❓ Question 5(f) ❓

Use the DBN to derive an expression for the Markov-chain transition probabilities
$$
P(X_{n+1}=j \mid X_n=i).
$$

Your answer should be written as a sum over possible demand values:
$$
P(X_{n+1}=j \mid X_n=i) = \sum_d P(X_{n+1}=j \mid X_n=i, D_n=d) P(D_n=d).
$$

Then compute the full transition matrix.

Write your answer in `answers.py:q5f`.


In [ ]:
Markdown(answers.q5f)

### ❓ Question 5(g) ❓

Compute the probability that the mission log reports insufficient cartridges in episode $n$, conditioned on the current state:
$$
P(O_n = 1 \mid X_n = i), \qquad i \in \{1,2,3\}.
$$

Express your answers numerically using the given Poisson probabilities.

Write your answer in `answers.py:q5g`.


In [ ]:
Markdown(answers.q5g)

### ❓ Question 5(h) ❓

Let $\pi$ be the stationary distribution of the Markov chain from Question 4.

Write an expression for the long-run probability that the mission log reports insufficient cartridges:
$$
P(O_n=1)=\sum_{i\in\{1,2,3\}} P(O_n=1 \mid X_n=i)\,\pi_i.
$$

Then evaluate this quantity numerically.

Write your answer in `answers.py:q5h`.


In [ ]:
Markdown(answers.q5h)

### ❓ Question 5(i) ❓

Briefly explain how this DBN extends the Markov-chain model from Question 4.

In particular:
- What extra information is represented by the node $O_n$?
- Why is this now naturally viewed as a dynamic Bayesian network rather than only as a Markov chain?
- How could a variable like $O_n$ be useful in a caldera mission where the true demand is not directly observed, but mission logs are available?

Write your answer in `answers.py:q5i`.


In [ ]:
Markdown(answers.q5i)